In [ ]:
import torch
import torchaudio
import soundfile as sf
from transformers import WhisperForConditionalGeneration, WhisperProcessor


MODEL_PATH = ...

device = "cuda:0"
model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH)
model.to(device)
processor = WhisperProcessor.from_pretrained(MODEL_PATH)


def postprocess_text(text: str) -> str:
    text = text.lower().strip()
    text = "".join(s for s in text if str(s).isalpha() or s == " ")
    return text


def infer_whisper(wav: torch.Tensor, sample_rate: int, lang: str = "en") -> str:
    # Обрабатываем аудио
    inputs = processor(
        wav.squeeze().numpy(),
        sampling_rate=sample_rate,
        return_tensors="pt",
    )

    # Генерируем транскрипцию
    outputs = model.generate(
        **inputs.to(device),
        language=f"<|{lang}|>"
    )

    hypothesis = processor.batch_decode(outputs, skip_special_tokens=True)[0]

    return hypothesis


# wav, sr = torchaudio.load("./data/audio/hello_avito.wav")
def transcribe(wav, sr):
    wav = torchaudio.transforms.Resample(sr, 16000)(wav)
    transcription = infer_whisper(wav, sample_rate=16000, lang="ru")
    return postprocess_text(transcription)
